In [ ]:
!pip install "tensorboard~=2.19.0" --quiet


In [ ]:
import tensorflow as tf, tensorboard as tb
print(tf.__version__)
print(tb.__version__)


2.19.0
2.19.0


In [ ]:
!pip install --upgrade pip --quiet
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer gradio datasets pyarrow --quiet


In [ ]:
!pip install -U torchaudio torchvision --quiet


In [ ]:
!pip install -U torchcodec torch --quiet

In [ ]:
!pip install transformers==4.52.0 --quiet

Reason for being yanked: <none given>


In [ ]:
import torch, torchaudio, torchvision
print(torch.__version__)
print(torchaudio.__version__)
print(torchvision.__version__)


2.9.1+cu128
2.9.1+cu128
0.24.1+cu128


In [ ]:

import datasets
print(datasets.__version__)



4.4.2


In [ ]:
import transformers
print(transformers.__version__)


4.52.0


In [ ]:
# =========================
# 1) Kurulum (Kaggle)
# =========================
# Eğer gerekirse çalıştır:
#!pip install -q transformers==4.37.2 torchaudio torch scikit-learn datasets

import os
import re
import torch
import torchaudio
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

In [ ]:
# ============================================================
# Unified SER Dataset Builder (RAVDESS + CREMA-D + TESS + SAVEE)
# Target labels: anger, disgust, fear, happy, sad, surprise
# Output: pandas DataFrame with columns ["audio_path", "emotion"]
# ============================================================

import os
import pandas as pd

# -------------------------
# 1) Target label space
# -------------------------
TARGET_LABELS = ["anger", "disgust", "fear", "happy", "sad", "surprise"]
label2id = {v: i for i, v in enumerate(TARGET_LABELS)}
id2label = {i: v for v, i in label2id.items()}

# -------------------------
# 2) Dataset-specific maps
# -------------------------
# TESS keys: angry, disgust, fear, happy, sad, ps (pleasant surprise)
tess_code2label = {
    "angry":   "anger",
    "disgust": "disgust",
    "fear":    "fear",
    "happy":   "happy",
    "sad":     "sad",
    "ps":      "surprise",
    # "neutral": excluded on purpose (not in TARGET_LABELS)
}

# RAVDESS emotion ids (3rd field in filename): 03 happy, 04 sad, 05 angry, 06 fear, 07 disgust, 08 surprise
ravdess_id2label = {
    "03": "happy",
    "04": "sad",
    "05": "anger",
    "06": "fear",
    "07": "disgust",
    "08": "surprise",
    # 01 neutral, 02 calm excluded
}

# CREMA-D emotion codes (3rd token in filename): ANG, DIS, FEA, HAP, SAD, SUR
crema_code2label = {
    "ANG": "anger",
    "DIS": "disgust",
    "FEA": "fear",
    "HAP": "happy",
    "SAD": "sad",
    "SUR": "surprise",
}

# SAVEE codes typically appear like: _a, _d, _f, _h, _s, _su
savee_code2label = {
    "a":  "anger",
    "d":  "disgust",
    "f":  "fear",
    "h":  "happy",
    "s":  "sad",
    "su": "surprise",
    # "n": neutral excluded if present
}

# -------------------------
# 3) Parsers per dataset
# -------------------------
def parse_tess(root: str):
    rows = []
    if not os.path.exists(root):
        print(f"[WARN] TESS path not found: {root}")
        return rows

    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(".wav"):
                base = f.lower()
                for k, v in tess_code2label.items():
                    if f"_{k}" in base:
                        rows.append([os.path.join(r, f), v])
                        break
    return rows


def parse_ravdess(root: str):
    rows = []
    if not os.path.exists(root):
        print(f"[WARN] RAVDESS path not found: {root}")
        return rows

    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(".wav"):
                parts = f.split("-")
                # expected: 03-01-05-02-02-01-12.wav -> parts[2] is emotion id
                if len(parts) >= 3:
                    emo_id = parts[2]
                    label = ravdess_id2label.get(emo_id)
                    if label is not None:
                        rows.append([os.path.join(r, f), label])
    return rows


def parse_crema(root: str):
    rows = []
    if not os.path.exists(root):
        print(f"[WARN] CREMA-D path not found: {root}")
        return rows

    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(".wav"):
                # example: 1001_IEO_HAP_HI.wav -> 3rd token is HAP
                parts = f.split("_")
                if len(parts) >= 3:
                    emo_code = parts[2].upper()
                    label = crema_code2label.get(emo_code)
                    if label is not None:
                        rows.append([os.path.join(r, f), label])
    return rows


def parse_savee(root: str):
    rows = []
    if not os.path.exists(root):
        print(f"[WARN] SAVEE path not found: {root}")
        return rows

    # IMPORTANT: check "su" before "s" (otherwise surprise may be misread as sad)
    keys_in_order = ["su", "a", "d", "f", "h", "s"]

    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(".wav"):
                name = f.lower()
                found = False
                for k in keys_in_order:
                    if f"_{k}" in name:
                        rows.append([os.path.join(r, f), savee_code2label[k]])
                        found = True
                        break
                if not found:
                    pass
    return rows


# -------------------------
# 4) Build unified DataFrame
# -------------------------
# Update this base path to your Drive folder:
DATA_ROOT = "/content/drive/MyDrive/Bitirme_projesi/archive-3"

# Update folder names if yours are different:
TESS_DIR    = os.path.join(DATA_ROOT, "Tess")
RAVDESS_DIR = os.path.join(DATA_ROOT, "Ravdess")
CREMA_DIR   = os.path.join(DATA_ROOT, "Crema")
SAVEE_DIR   = os.path.join(DATA_ROOT, "Savee")

all_rows = []
all_rows += parse_tess(TESS_DIR)
all_rows += parse_ravdess(RAVDESS_DIR)
all_rows += parse_crema(CREMA_DIR)
all_rows += parse_savee(SAVEE_DIR)

df = pd.DataFrame(all_rows, columns=["audio_path", "emotion"])

print("BİRLEŞİK DATASET - Örnek satırlar:")
print(df.head())

print("\nBİRLEŞİK DATASET - Sınıf dağılımı:")
print(df["emotion"].value_counts())

print("\nToplam örnek sayısı:", len(df))
print("Etiket kümesi:", sorted(df["emotion"].unique().tolist()))


BİRLEŞİK DATASET - Örnek satırlar:
                                          audio_path  emotion
0  /content/drive/MyDrive/Bitirme_projesi/archive...  disgust
1  /content/drive/MyDrive/Bitirme_projesi/archive...  disgust
2  /content/drive/MyDrive/Bitirme_projesi/archive...  disgust
3  /content/drive/MyDrive/Bitirme_projesi/archive...  disgust
4  /content/drive/MyDrive/Bitirme_projesi/archive...  disgust

BİRLEŞİK DATASET - Sınıf dağılımı:
emotion
disgust     1923
happy       1923
sad         1923
anger       1923
fear        1922
surprise     652
Name: count, dtype: int64

Toplam örnek sayısı: 10266
Etiket kümesi: ['anger', 'disgust', 'fear', 'happy', 'sad', 'surprise']


In [ ]:
# =========================
# 4) Stratified Train/Valid
# =========================
train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["emotion"],
    random_state=42
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_ds = Dataset.from_pandas(valid_df.reset_index(drop=True))
raw_ds = DatasetDict({"train": train_ds, "validation": valid_ds})

In [ ]:
print(train_df)

                                             audio_path  emotion
5469  /content/drive/MyDrive/Bitirme_projesi/archive...    anger
5863  /content/drive/MyDrive/Bitirme_projesi/archive...  disgust
5509  /content/drive/MyDrive/Bitirme_projesi/archive...    happy
1392  /content/drive/MyDrive/Bitirme_projesi/archive...    happy
4716  /content/drive/MyDrive/Bitirme_projesi/archive...     fear
...                                                 ...      ...
7803  /content/drive/MyDrive/Bitirme_projesi/archive...    happy
7398  /content/drive/MyDrive/Bitirme_projesi/archive...      sad
7134  /content/drive/MyDrive/Bitirme_projesi/archive...     fear
9590  /content/drive/MyDrive/Bitirme_projesi/archive...    happy
2944  /content/drive/MyDrive/Bitirme_projesi/archive...  disgust

[8212 rows x 2 columns]


In [ ]:
raw_ds['train'][0]

{'audio_path': '/content/drive/MyDrive/Bitirme_projesi/archive-3/Crema/1060_ITH_ANG_XX.wav',
 'emotion': 'anger'}

In [ ]:
from datasets import Dataset, Audio, ClassLabel


In [ ]:
raw_ds['train'][0]

{'audio_path': '/content/drive/MyDrive/Bitirme_projesi/archive-3/Crema/1060_ITH_ANG_XX.wav',
 'emotion': 'anger'}

In [ ]:
# =========================
# 5) Feature Extractor
# =========================
model_name = "facebook/wav2vec2-large-xlsr-53"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

TARGET_SR = 16000
MAX_SEC   = 6
MAX_LEN   = TARGET_SR * MAX_SEC  # 96000 örnek

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

In [ ]:
from datasets import Audio

raw_ds = raw_ds.rename_column("audio_path", "audio")
raw_ds = raw_ds.cast_column("audio", Audio(sampling_rate=16000))


In [ ]:


def add_label(batch):
    batch["label"] = label2id[batch["emotion"]]
    return batch

raw_ds = raw_ds.map(add_label)

Map:   0%|          | 0/8212 [00:00<?, ? examples/s]

Map:   0%|          | 0/2054 [00:00<?, ? examples/s]

In [ ]:
a = raw_ds["train"][0]["audio"]
arr = a["array"]
sr = a["sampling_rate"]
print(type(arr), arr.shape, sr)


<class 'numpy.ndarray'> (42709,) 16000


In [ ]:
raw_ds

DatasetDict({
    train: Dataset({
        features: ['audio', 'emotion', 'label'],
        num_rows: 8212
    })
    validation: Dataset({
        features: ['audio', 'emotion', 'label'],
        num_rows: 2054
    })
})

In [ ]:
import numpy as np
import torch

def decode_audio_any(a):
    # 1) dict gibi davranıyorsa
    try:
        return a["array"], a["sampling_rate"]
    except Exception:
        pass

    # 2) torchcodec AudioDecoder ise
    if hasattr(a, "get_all_samples"):
        samples = a.get_all_samples()  # AudioSamples
        # AudioSamples içinden tensor/sr çekmeye çalış (sürüm farkları olabilir)
        for data_attr in ["data", "samples", "tensor"]:
            if hasattr(samples, data_attr):
                x = getattr(samples, data_attr)
                break
        else:
            x = samples  # son çare

        # sampling rate alanı
        sr = None
        for sr_attr in ["sample_rate", "sampling_rate", "rate"]:
            if hasattr(samples, sr_attr):
                sr = int(getattr(samples, sr_attr))
                break
        if sr is None and hasattr(a, "sample_rate"):
            sr = int(getattr(a, "sample_rate"))

        # tensor -> numpy
        if isinstance(x, torch.Tensor):
            x = x.detach().cpu().numpy()
        x = np.asarray(x)

        # stereo ise mono
        if x.ndim == 2:
            # (channels, time) veya (time, channels) olabilir
            if x.shape[0] <= 8:
                x = x.mean(axis=0)
            else:
                x = x.mean(axis=1)

        return x, sr

    raise TypeError(f"Unsupported audio type: {type(a)}")


In [ ]:
a = raw_ds["train"][10]["audio"]
x, sr = decode_audio_any(a)
print(x.shape, sr, x.dtype)


(29896,) 16000 float32


In [ ]:
def preprocess_fast(batch):
    audio_arrays = []
    for a in batch["audio"]:
        x, sr = decode_audio_any(a)
        audio_arrays.append(x)

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,  # cast_column ile 16k yapıyorsan sabit ver
        padding=True,
        return_tensors="pt"
    )
    inputs["labels"] = batch["label"]
    return inputs


encoded = raw_ds.map(
    preprocess_fast,
    batched=True,
    num_proc=4,  # Colab T4 için iyi
    remove_columns=raw_ds["train"].column_names
)


Map (num_proc=4):   0%|          | 0/8212 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2054 [00:00<?, ? examples/s]

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
from transformers import Wav2Vec2ForSequenceClassification

num_labels = 6

base_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
)

base_model.to(device)
base_model.eval()


Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-large-xlsr-53 and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2ForSequenceClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=

In [ ]:
import torch

def data_collator(features):
    # features: list of dicts, each has input_values, attention_mask, labels
    batch = feature_extractor.pad(
        [
            {
                "input_values": f["input_values"],
                "attention_mask": f["attention_mask"],
            }
            for f in features
        ],
        padding=True,
        return_tensors="pt",
    )
    batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
    return batch


In [ ]:
from transformers import Trainer

base_trainer = Trainer(
    model=base_model,
    args=args,              # aynı args, ama train etmeyeceğiz
    eval_dataset=encoded["validation"],    # AYNI test seti
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:
base_metrics = base_trainer.evaluate()
print(base_metrics)


{'eval_loss': 1.793819785118103, 'eval_model_preparation_time': 0.0052, 'eval_accuracy': 0.18743914313534565, 'eval_f1_macro': 0.052617192838595055, 'eval_runtime': 205.6862, 'eval_samples_per_second': 9.986, 'eval_steps_per_second': 2.499}


In [ ]:
print(raw_ds)

DatasetDict({
    train: Dataset({
        features: ['audio', 'emotion', 'label'],
        num_rows: 8212
    })
    validation: Dataset({
        features: ['audio', 'emotion', 'label'],
        num_rows: 2054
    })
})


In [ ]:
encoded["train"][10]

{'input_values': [-0.35768648982048035,
  -0.43140819668769836,
  -0.422472208738327,
  -0.36438846588134766,
  -0.3308786153793335,
  -0.3197086453437805,
  -0.28619879484176636,
  -0.36438846588134766,
  -0.2459869533777237,
  -0.1677972972393036,
  -0.17003127932548523,
  -0.22588104009628296,
  -0.1141815260052681,
  -0.17003127932548523,
  -0.18120123445987701,
  -0.04939580708742142,
  -0.10301157832145691,
  -0.03822585195302963,
  -0.0091839749366045,
  0.08464362472295761,
  0.14719536900520325,
  0.20751310884952545,
  0.3728283941745758,
  0.3594244420528412,
  0.4800599217414856,
  0.5649515390396118,
  0.6543111801147461,
  0.741436779499054,
  0.8307964205741882,
  0.8017545342445374,
  0.8799442052841187,
  0.8598383069038391,
  0.826328456401825,
  0.8866462111473083,
  0.88888019323349,
  0.9201560616493225,
  0.8196264505386353,
  0.7861166000366211,
  0.7660107016563416,
  0.7883505821228027,
  0.8486683368682861,
  0.8844121694564819,
  0.7816486358642578,
  0.84196

In [ ]:
print(encoded)

DatasetDict({
    train: Dataset({
        features: ['input_values', 'attention_mask', 'labels'],
        num_rows: 8213
    })
    validation: Dataset({
        features: ['input_values', 'attention_mask', 'labels'],
        num_rows: 2054
    })
})


In [ ]:
# =========================
# 7) Model
# =========================
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label,
    problem_type="single_label_classification"
)

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-large-xlsr-53 and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# =========================
# 8) Metrikler
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1m}

In [ ]:
print(encoded["train"].column_names)
print(encoded["train"][0].keys())


['input_values', 'attention_mask', 'labels']
dict_keys(['input_values', 'attention_mask', 'labels'])


In [ ]:
encoded["train"][0]

{'input_values': [0.05119147524237633,
  0.05682031437754631,
  0.057445742189884186,
  0.06244915351271629,
  0.07308140397071838,
  0.06432542949914932,
  0.06682714074850082,
  0.07808481901884079,
  0.05119147524237633,
  0.0518169030547142,
  0.022421853616833687,
  0.007411617320030928,
  0.009913323447108269,
  0.0005319251795299351,
  0.008662470616400242,
  -0.019481724128127098,
  -0.015103737823665142,
  -0.02385971136391163,
  0.003033631481230259,
  -0.011351178400218487,
  0.0017827783012762666,
  -0.0057223401963710785,
  -0.01635459065437317,
  -0.009474899619817734,
  0.014916735701262951,
  0.011789603158831596,
  -0.0025952074211090803,
  -0.01635459065437317,
  -0.013227458111941814,
  0.003033631481230259,
  0.025548985227942467,
  0.030552398413419724,
  0.016167588531970978,
  0.011164176277816296,
  0.016793014481663704,
  0.012415029108524323,
  0.009913323447108269,
  -0.020107151940464973,
  -0.03449196368455887,
  -0.03699366748332977,
  -0.03011397644877433

In [ ]:
# =========================
# 9) TrainingArguments & Trainer
# =========================
output_dir = "./savee_wav2vec2_results"

args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,          # SES için genelde 1e-5 ~ 5e-5 arası iyi çalışır
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",            # TensorBoard istersen "tensorboard"
)




In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipython-input-3927935291.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.540100,1.526884,0.395813,0.311531
2,0.982900,1.000315,0.629503,0.646553
3,0.761500,0.794248,0.719572,0.736406
4,0.637600,0.814296,0.753165,0.760500
5,0.796800,0.765341,0.776047,0.789482
6,0.668000,0.757643,0.804284,0.814624
7,0.637400,0.786031,0.798442,0.812491
8,0.655200,0.711479,0.817916,0.831046
9,0.500900,0.697583,0.825706,0.837521
10,0.656200,0.682195,0.835930,0.847350


TrainOutput(global_step=20530, training_loss=0.8123606266071898, metrics={'train_runtime': 15873.3492, 'train_samples_per_second': 5.173, 'train_steps_per_second': 1.293, 'total_flos': 1.7264357854609238e+19, 'train_loss': 0.8123606266071898, 'epoch': 10.0})

In [ ]:

# =========================
# 10) Kaydetme
# =========================
final_dir = os.path.join(output_dir, "/content/drive/MyDrive/Bitirme_projesi/Modeller/wav2vec2_son")
model.save_pretrained(final_dir)
feature_extractor.save_pretrained(final_dir)

print("Eğitim tamamlandı. Model ve feature_extractor şurada:", final_dir)


Eğitim tamamlandı. Model ve feature_extractor şurada: /content/drive/MyDrive/Bitirme_projesi/Modeller/wav2vec2_son


In [ ]:
import torch, torchaudio
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

MODEL_DIR = "/content/drive/MyDrive/Bitirme_projesi/Modeller/wav2vec2_son"  # <-- değiştir
AUDIO_PATH = "/content/drive/MyDrive/Bitirme_projesi/dataset/deneme1.wav"      # <-- değiştir

# label setin (6 sınıf)
TARGET_LABELS = ["anger","disgust","fear","happy","sad","surprise"]
id2label = {i:l for i,l in enumerate(TARGET_LABELS)}
label2id = {l:i for i,l in id2label.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model + extractor (fine-tuned)
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=len(TARGET_LABELS),
    id2label=id2label,
    label2id=label2id,
).to(device).eval()

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_DIR)

def load_16k_mono(path, target_sr=16000):
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
        sr = target_sr
    return wav.squeeze(0), sr

wav, sr = load_16k_mono(AUDIO_PATH, 16000)

inputs = feature_extractor(
    wav.numpy(),
    sampling_rate=sr,
    return_tensors="pt",
    padding=True
)

with torch.no_grad():
    logits = model(
        input_values=inputs["input_values"].to(device),
        attention_mask=inputs.get("attention_mask", None).to(device) if "attention_mask" in inputs else None
    ).logits

probs = torch.softmax(logits, dim=-1).squeeze(0).cpu()
top_id = int(torch.argmax(probs))
print("Prediction:", id2label[top_id])
print("Probs:", {id2label[i]: float(probs[i]) for i in range(len(TARGET_LABELS))})


Prediction: sad
Probs: {'anger': 4.813019040739164e-05, 'disgust': 0.0048753321170806885, 'fear': 0.02538994327187538, 'happy': 0.007650104817003012, 'sad': 0.9570974707603455, 'surprise': 0.0049389866180717945}
